In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from autofocus import Autofocus

os.environ["OMERO_HOST"] = "100.125.247.59"
os.environ["OMERO_PORT"] = "4064"
os.environ["OMERO_USERNAME"] = "root"
os.environ["OMERO_PASSWORD"] = "omero"

In [5]:
def is_substrate_background(
    image,
    hue_range=(110, 170),
    min_saturation=30,
    min_purple_fraction=0.4,
    sample_fraction=0.5,
):
    """Check whether the image background is a purple SiO2 substrate.

    Returns True when the majority of the central region has purple hue
    with sufficient saturation (i.e. good for flake hunting).  Returns
    False for grey / brown bare-metal backgrounds.

    Args:
        image: BGR uint8 image (OpenCV format).
        hue_range: (lo, hi) acceptable hue in OpenCV 0-180 scale.
                    Default (110, 170) covers violet through magenta.
        min_saturation: pixels below this S value are considered grey.
        min_purple_fraction: fraction of sampled pixels that must be
                                purple for the image to pass.
        sample_fraction: fraction of the image (centred) to sample,
                            avoids vignetting at edges.

    Returns:
        (bool, dict) – pass/fail and diagnostic stats.
    """
    h, w = image.shape[:2]
    margin_y = int(h * (1 - sample_fraction) / 2)
    margin_x = int(w * (1 - sample_fraction) / 2)
    roi = image[margin_y:h - margin_y, margin_x:w - margin_x]

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    hue = hsv[:, :, 0].ravel().astype(np.float32)
    sat = hsv[:, :, 1].ravel().astype(np.float32)

    saturated = sat >= min_saturation
    in_hue = (hue >= hue_range[0]) & (hue <= hue_range[1])
    purple_mask = saturated & in_hue

    n_total = len(hue)
    n_purple = int(purple_mask.sum())
    purple_frac = n_purple / max(n_total, 1)

    stats = {
        "median_hue": float(np.median(hue)),
        "median_saturation": float(np.median(sat)),
        "purple_fraction": round(purple_frac, 4),
        "n_pixels_sampled": n_total,
        "passed": purple_frac >= min_purple_fraction,
    }
    return stats["passed"], stats

def get_color_features(image, saturation_threshold=40):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    sat = hsv[:, :, 1]
    color_mask  = sat > saturation_threshold
    colorful_pixels = np.count_nonzero(color_mask)
    total_pixels = image.shape[0] * image.shape[1]

    color_ratio = colorful_pixels / total_pixels
    return color_ratio

def exist_color_features(image, ratio_threshold=0.05):
    color_ratio = Autofocus.get_color_features(image)
    return color_ratio >= ratio_threshold

In [7]:
image_dir = Path("images")
good_images = sorted(image_dir.glob("image[0-9]*.png"))
bad_images = sorted(image_dir.glob("image_bad[0-5]*.png"))

print(f"Good images: {[p.name for p in good_images]}")
print(f"Bad images:  {[p.name for p in bad_images]}")

results = []
for path in good_images + bad_images:
    img = cv2.imread(str(path))
    if img is None:
        print(f"  SKIP (could not read) {path.name}")
        continue
    passed, stats = is_substrate_background(img)
    exists = exist_color_features(img)
    label = "GOOD" if path in good_images else "BAD"
    result_str = "PASS" if passed else "FAIL"
    results.append((path.name, label, result_str, stats))
    print(f"  {path.name:20s}  expected={label:4s}  result={result_str:4s}  "
          f"hue={stats['median_hue']:.0f}  sat={stats['median_saturation']:.0f}  "
          f"purple={stats['purple_fraction']:.2%}  exists={exists}")
all_correct = all(
    (label == "GOOD" and result == "PASS") or (label == "BAD" and result == "FAIL")
    for _, label, result, _ in results
)
print(f"\nAll classifications correct: {all_correct}")

Good images: ['image1.png', 'image10.png', 'image2.png', 'image3.png', 'image4.png', 'image5.png', 'image6.png', 'image7.png', 'image8.png', 'image9.png']
Bad images:  ['image_bad1.png', 'image_bad2.png', 'image_bad3.png', 'image_bad4.png', 'image_bad5.png']
  image1.png            expected=GOOD  result=PASS  hue=148  sat=74  purple=96.28%  exists=True
  image10.png           expected=GOOD  result=PASS  hue=142  sat=75  purple=85.46%  exists=True
  image2.png            expected=GOOD  result=PASS  hue=150  sat=80  purple=96.74%  exists=True
  image3.png            expected=GOOD  result=PASS  hue=146  sat=75  purple=99.68%  exists=True
  image4.png            expected=GOOD  result=PASS  hue=148  sat=74  purple=96.90%  exists=True
  image5.png            expected=GOOD  result=PASS  hue=148  sat=75  purple=100.00%  exists=True
  image6.png            expected=GOOD  result=PASS  hue=147  sat=74  purple=98.79%  exists=True
  image7.png            expected=GOOD  result=PASS  hue=149  sat=75 